In [1]:
##The script should make rlon correction, create corner coordinate for NarClim2.0 rotated grid and regrid using conservative regridding

#!check if this script corrects for rlon as instructed by NarClim2.0 readme if there is any further issue 
#!/g/data/gh02/dev/environments/intel_general/20211104/bin/python3.9
from pathlib import Path
import xarray as xr
import numpy as np
import cartopy.crs as ccrs 
import xesmf

import pandas as pd
import glob
import warnings
import os


warnings.filterwarnings("ignore", category=UserWarning) #mute warnings 

In [2]:

# target grid details --AUS18 
# need to be changes if regridding 4km res SE Australia region 
target_grid_details={
    'lon':(
        112, # 136, min longitude (corner of corner point)
        155, # 158, max longitude (corner of corner point)
        216, # 441, number of grid points + 1
    ),
    'lat':(
        -45, # min latitude (corner of corner point)
        -9, # max latitude (corner of corner point)
        181, # number of grid points + 1
    )
}

In [3]:

run_path=Path('/scratch/dt6/xw9650/tmp').resolve()
base_path=run_path.parent

# Output file
output_base=run_path.joinpath('regridded')
output_base.mkdir(exist_ok=True)


In [4]:

# Projections
plate_caree=ccrs.PlateCarree()
rot_pole=ccrs.RotatedPole(
    pole_longitude=141.38,
    pole_latitude=60.31,
    central_rotated_longitude=141.38
)

In [5]:

# Example target grid details

# don't change this function if you don't know what it does...
# Used only to getnerate target grid from above dictionary
def generate_grid(grid_dict):
    lons=np.linspace(*grid_dict['lon'])
    lats=np.linspace(*grid_dict['lat'])
    return lons,lats

# don't change this function if you don't know what it does...
# Finds corners of rotated pole coordinates in rotated pole projection
def make_mid_points(x,dx):
    out=x-dx
    return np.append(out,dx+x[-1])

# don't change this function if you don't know what it does...
# Finds corners of rotated pole coordinates in plate-caree projection
def make_lat_lon_b(geo_lon,geo_lat):
    """a little more complicated than it would have needed
    to be if rlon was calculated correctly in the first place...
    """
    tmp = rot_pole.transform_points(plate_caree, geo_lon.values, geo_lat.values)
    rlon = tmp[:, :, 0]
    rlat = tmp[:, :, 1]

    rlon_1d=np.mean(rlon,axis=0)
    rlat_1d=np.mean(rlat,axis=1)
    dx=0.176
    rlonb_1d=make_mid_points(rlon_1d,0.5*dx)
    rlatb_1d=make_mid_points(rlat_1d,0.5*dx)

    rlonb=np.asarray(tuple(map(lambda x:rlonb_1d,rlatb_1d)))
    rlatb=np.asarray(tuple(map(lambda x:rlatb_1d,rlonb_1d))).transpose()

    tmp = plate_caree.transform_points(rot_pole, rlonb, rlatb)
    lon_b = tmp[:, :, 0]
    lat_b = tmp[:, :, 1]

    lon_b=xr.DataArray(lon_b,coords={'rlat_b':rlatb_1d,'rlon_b':rlonb_1d})
    lat_b=xr.DataArray(lat_b,coords={'rlat_b':rlatb_1d,'rlon_b':rlonb_1d})

    return lon_b, lat_b

# don't change this function if you don't know what it does...
# Uses generate grid to create target corner grids and uses that information
# to find mid points of grid
def create_target_grid():
    tlon_b,tlat_b=generate_grid(target_grid_details)
    tlon=0.5*(tlon_b[:-1]+tlon_b[1:])
    tlat=0.5*(tlat_b[:-1]+tlat_b[1:])
    dummy=np.zeros((len(tlon),len(tlat)))
    target_grid=xr.DataArray(dummy,dims=['lon','lat']).to_dataset(name='target_grid')
    return target_grid.assign_coords({'lon':tlon,'lat':tlat,'lon_b':tlon_b,'lat_b':tlat_b}).transpose()

# don't change this function if you don't know what it does...
# Performs the regridding
def run_regrid(ds,target):
    regridder=xesmf.Regridder(
            ds,
            target,
            'conservative',
            reuse_weights=False
        )
    
    regridded=regridder(ds,keep_attrs=True)
    return regridded.drop_vars(('lon_b','lat_b','rotated_pole','rlat_b','rlon_b'),errors='ignore')

# main function
def main():
    ds=xr.load_dataset(file) # open netcdf file
    slon_b, slat_b=make_lat_lon_b(ds.lon,ds.lat)
    ds=ds.assign_coords({'lon_b':np.mod(slon_b,360),'lat_b':slat_b})
    ds['lon']=np.mod(ds.lon,360)
    target_grid=create_target_grid()
    regridded_ds=run_regrid(ds,target_grid)


    name=file.name
    output_name=output_base.joinpath(name)
    regridded_ds=regridded_ds.to_netcdf(output_name)

In [6]:
#standardised run
def find_input_files(var, gcm, scenario, member, model, run, domain, institute):
    base_path = f"/g/data/zz63/NARCliM2-0/output-CMIP6/DD/{domain}/{institute}/{gcm}/"
    path_mask = (
        f"{scenario}/{member}/{model}/{run}/day/{var}/latest/"
        f"{var}_{domain}_{gcm}_{scenario}_{member}_{institute}_{model}_{run}_day_*.nc"
    )
    return glob.glob(os.path.join(base_path, path_mask))


In [7]:
#pre-release 
def find_input_files_prerelease(var, gcm, scenario, member, model, run, domain, institute):
    base_path = f"/g/data/zz63/NARCliM2-0-derived/output-CMIP6/DD/{domain}/{institute}/{gcm}/"
    path_mask = (
        f"{scenario}/{member}/{model}/{run}/day/{var}/latest/"
        f"{var}_{domain}_{gcm}_{scenario}_{member}_{institute}_{model}_{run}_day_*.nc"
    )
    return glob.glob(os.path.join(base_path, path_mask))

In [7]:
#ACCESS iteration
# Define all combinations you want to iterate
variables = ['hurs','tas','pr']
gcms = ['ACCESS-ESM1-5']  
scenarios = ['ssp370','historical','ssp126','ssp245']
domains = ['AUS-18']
members = ['r6i1p1f1']# ! this is different between GCMs 
models = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']
runs = ['v1-r1']  
institutes = ['NSW-Government']


# Main loop over all combinations
for var in variables:
    for gcm in gcms:
        for scenario in scenarios:
            for member in members:
                for model in models:
                    for run in runs:
                        for domain in domains:
                            for institute in institutes:
                                files = find_input_files(var, gcm, scenario, member, model, run, domain, institute)
                                if files:
                                    print(f"Processing: {var}, {gcm}, {scenario}, {member}")
                                    input_example_file = [Path(file) for file in files]
                                    
                                    for file in input_example_file:
                                        main()  # Pass file if your main() accepts it
                                else:
                                    print(f"No files found for: {var}, {gcm}, {scenario}, {member}")


Processing: hurs, ACCESS-ESM1-5, ssp370, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, ssp370, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, historical, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, historical, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, ssp126, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, ssp126, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, ssp245, r6i1p1f1
Processing: hurs, ACCESS-ESM1-5, ssp245, r6i1p1f1


In [8]:
# EC-Earth3-Veg iteration 
# Define all combinations you want to iterate
variables = ['hurs','tas','pr']
gcms = ['EC-Earth3-Veg']  
scenarios = [ 'historical', 'ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f1']# ! this is different between GCMs 
models = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']
runs = ['v1-r1']  
institutes = ['NSW-Government']


# Main loop over all combinations
for var in variables:
    for gcm in gcms:
        for scenario in scenarios:
            for member in members:
                for model in models:
                    for run in runs:
                        for domain in domains:
                            for institute in institutes:
                                files = find_input_files(var, gcm, scenario, member, model, run, domain, institute)
                                if files:
                                    print(f"Processing: {var}, {gcm}, {scenario}, {member}")
                                    input_example_file = [Path(file) for file in files]
                                    
                                    for file in input_example_file:
                                        main()  # Pass file if your main() accepts it
                                else:
                                    print(f"No files found for: {var}, {gcm}, {scenario}, {member}")


Processing: hurs, EC-Earth3-Veg, historical, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, historical, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, ssp126, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, ssp126, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, ssp245, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, ssp245, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, ssp370, r1i1p1f1
Processing: hurs, EC-Earth3-Veg, ssp370, r1i1p1f1


In [ ]:
# MPI iteration 
# Define all combinations you want to iterate
variables = ['hurs','tas','pr']
gcms = ['MPI-ESM1-2-HR'] 
scenarios = [ 'historical','ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f1']# ! this is different between GCMs 
models = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']
runs = ['v1-r1'] 
institutes = ['NSW-Government']


# Main loop over all combinations
for var in variables:
    for gcm in gcms:
        for scenario in scenarios:
            for member in members:
                for model in models:
                    for run in runs:
                        for domain in domains:
                            for institute in institutes:
                                files = find_input_files(var, gcm, scenario, member, model, run, domain, institute)
                                if files:
                                    print(f"Processing: {var}, {gcm}, {scenario}, {member}")
                                    input_example_file = [Path(file) for file in files]
                                    
                                    for file in input_example_file:
                                        main()  # Pass file if your main() accepts it
                                else:
                                    print(f"No files found for: {var}, {gcm}, {scenario}, {member}")


Processing: hurs, MPI-ESM1-2-HR, historical, r1i1p1f1


In [ ]:
# NorESM2-MM iteration 
# Define all combinations you want to iterate
variables = ['hurs','tas','pr']
gcms = ['NorESM2-MM']  
scenarios = ['historical','ssp126','ssp245', 'ssp370']
domains = ['AUS-18']
members = ['r1i1p1f1']# ! this is different between GCMs 
models = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']
runs = ['v1-r1']  
institutes = ['NSW-Government']


# Main loop over all combinations
for var in variables:
    for gcm in gcms:
        for scenario in scenarios:
            for member in members:
                for model in models:
                    for run in runs:
                        for domain in domains:
                            for institute in institutes:
                                files = find_input_files(var, gcm, scenario, member, model, run, domain, institute)
                                if files:
                                    print(f"Processing: {var}, {gcm}, {scenario}, {member}")
                                    input_example_file = [Path(file) for file in files]
                                    
                                    for file in input_example_file:
                                        main()  # Pass file if your main() accepts it
                                else:
                                    print(f"No files found for: {var}, {gcm}, {scenario}, {member}")


In [ ]:
# UKESM1-0-LL iteration 
# Define all combinations you want to iterate
variables = ['hurs','tas','pr']
gcms = ['UKESM1-0-LL'] 
scenarios = ['historical','ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f2']# ! this is different between GCMs 
models = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']
runs = ['v1-r1']  
institutes = ['NSW-Government']


# Main loop over all combinations
for var in variables:
    for gcm in gcms:
        for scenario in scenarios:
            for member in members:
                for model in models:
                    for run in runs:
                        for domain in domains:
                            for institute in institutes:
                                files = find_input_files(var, gcm, scenario, member, model, run, domain, institute)
                                if files:
                                    print(f"Processing: {var}, {gcm}, {scenario}, {member}")
                                    input_example_file = [Path(file) for file in files]
                                    
                                    for file in input_example_file:
                                        main()  # Pass file if your main() accepts it
                                else:
                                    print(f"No files found for: {var}, {gcm}, {scenario}, {member}")
